# M1 — Backtest Visualization

Visualises the rolling-origin backtest results against the German TSO day-ahead forecast.

**What you should see:**
- The TSO baseline lands around **3.7% MAPE / 505 MW MAE** on the 2025-01 → 2026-02 evaluation window.
- Seasonal-naive (last week's same QH) is materially worse over the same window (skill score around -0.4).
- This is the bar future TF models in M4+ have to clear.

In [1]:
from datetime import date
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from loadforecast.backtest import (
    load_smard_15min,
    run_backtest,
    seasonal_naive_predict,
    tso_baseline_predict,
)

PARQUET = Path("../smard_merged_15min.parquet")
assert PARQUET.exists(), "Run from repo root or adjust path."

df = load_smard_15min(PARQUET)
print(f"Loaded {df.shape[0]:,} rows, {df.shape[1]} cols, {df.index.min()} -> {df.index.max()}")

Loaded 152,160 rows, 35 cols, 2022-01-01 00:00:00+00:00 -> 2026-05-04 23:45:00+00:00


## 1. Run both baselines over the evaluation window

In [2]:
EVAL_START = date(2025, 1, 1)
EVAL_END = date(2026, 2, 28)

res_tso = run_backtest(tso_baseline_predict, df, EVAL_START, EVAL_END, progress=False, label="tso")
res_naive = run_backtest(seasonal_naive_predict, df, EVAL_START, EVAL_END, progress=False, label="naive")

summary = pd.DataFrame(
    {
        "TSO baseline": res_tso.overall,
        "Seasonal-naive": res_naive.overall,
    }
).round(3)
summary

,TSO baseline,Seasonal-naive
mae_model,503.870,701.838
mae_tso,503.870,503.870
rmse_model,644.386,1050.172
rmse_tso,644.386,644.386
mape_model,3.731,5.155
mape_tso,3.731,3.731
skill_score,0.000,-0.393
n_days,424.000,424.000


## 2. Forecast vs actual — a sample week

In [3]:
SAMPLE_START = pd.Timestamp("2025-09-15", tz="Europe/Berlin").tz_convert("UTC")
SAMPLE_END = SAMPLE_START + pd.Timedelta(days=7)

wk_tso = res_tso.per_step.copy()
wk_tso["target_ts"] = pd.to_datetime(wk_tso["target_ts"], utc=True)
wk_tso = wk_tso[(wk_tso["target_ts"] >= SAMPLE_START) & (wk_tso["target_ts"] < SAMPLE_END)]

wk_naive = res_naive.per_step.copy()
wk_naive["target_ts"] = pd.to_datetime(wk_naive["target_ts"], utc=True)
wk_naive = wk_naive[(wk_naive["target_ts"] >= SAMPLE_START) & (wk_naive["target_ts"] < SAMPLE_END)]

fig = go.Figure()
fig.add_trace(go.Scatter(x=wk_tso["target_ts"], y=wk_tso["y_true"], name="Actual", line=dict(color="black", width=2)))
fig.add_trace(go.Scatter(x=wk_tso["target_ts"], y=wk_tso["y_tso"], name="TSO forecast", line=dict(color="#1f77b4", dash="dash")))
fig.add_trace(go.Scatter(x=wk_naive["target_ts"], y=wk_naive["y_model"], name="Seasonal-naive", line=dict(color="#d62728", dash="dot")))
fig.update_layout(
    title="Day-ahead forecast vs actual — sample week (Sep 15–22, 2025)",
    xaxis_title="Time (UTC)",
    yaxis_title="Grid load (MW)",
    hovermode="x unified",
    height=450,
)
fig

## 3. Daily MAE — TSO vs seasonal-naive over the full window

In [4]:
daily = res_tso.per_day[["issue_date", "mae_tso"]].merge(
    res_naive.per_day[["issue_date", "mae_model"]].rename(columns={"mae_model": "mae_naive"}),
    on="issue_date",
)
daily["issue_date"] = pd.to_datetime(daily["issue_date"])

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily["issue_date"], y=daily["mae_tso"], name="TSO MAE", line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=daily["issue_date"], y=daily["mae_naive"], name="Seasonal-naive MAE", line=dict(color="#d62728")))
fig.update_layout(
    title="Per-day MAE — German day-ahead consumption forecast",
    xaxis_title="Delivery date",
    yaxis_title="MAE (MW)",
    hovermode="x unified",
    height=450,
)
fig

## 4. Error distributions

In [5]:
err_tso = res_tso.per_step["y_true"] - res_tso.per_step["y_tso"]
err_naive = res_naive.per_step["y_true"] - res_naive.per_step["y_model"]

fig = make_subplots(rows=1, cols=2, subplot_titles=("TSO error (MW)", "Seasonal-naive error (MW)"))
fig.add_trace(go.Histogram(x=err_tso, nbinsx=80, name="TSO", marker_color="#1f77b4"), row=1, col=1)
fig.add_trace(go.Histogram(x=err_naive, nbinsx=80, name="Naive", marker_color="#d62728"), row=1, col=2)
fig.update_layout(
    title="Forecast error distributions (actual - forecast)",
    showlegend=False,
    height=400,
)
fig

## 5. MAE by hour-of-day — where does the TSO struggle?

In [6]:
ps = res_tso.per_step.copy()
ps["target_ts"] = pd.to_datetime(ps["target_ts"], utc=True)
ps["hour_local"] = ps["target_ts"].dt.tz_convert("Europe/Berlin").dt.hour
ps["abs_err"] = (ps["y_true"] - ps["y_tso"]).abs()
by_hour = ps.groupby("hour_local")["abs_err"].mean().reset_index()

fig = go.Figure(go.Bar(x=by_hour["hour_local"], y=by_hour["abs_err"], marker_color="#1f77b4"))
fig.update_layout(
    title="TSO forecast MAE by hour-of-day (Berlin local time)",
    xaxis_title="Hour",
    yaxis_title="Mean absolute error (MW)",
    height=400,
)
fig

## 6. Headline — the bar to beat

From this point forward, every model trained in M4+ is judged by:

$$\text{skill} = 1 - \frac{\text{MAE}_{\text{model}}}{\text{MAE}_{\text{TSO}}}$$

The MAE we have to beat (over Jan 2025 → Feb 2026, 424 days):

In [7]:
print(f"  TSO MAE  : {res_tso.overall['mae_model']:.1f} MW")
print(f"  TSO MAPE : {res_tso.overall['mape_model']:.2f}%")
print()
print(f"  Stretch target (15% MAPE reduction): {res_tso.overall['mape_model'] * 0.85:.2f}%")
print(f"  Stretch target MAE                 : {res_tso.overall['mae_model'] * 0.85:.1f} MW")

  TSO MAE  : 503.9 MW
  TSO MAPE : 3.73%

  Stretch target (15% MAPE reduction): 3.17%
  Stretch target MAE                 : 428.3 MW
